In [7]:
!pip install xgboost

  Using cached xgboost-3.0.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.0.0-py3-none-win_amd64.whl (150.0 MB)


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
import joblib

df = pd.read_csv('final_soil_dataset.csv')

print("Data shape:", df.shape)
df.head(2)

Data shape: (14700, 21)


,image_name,image_color,Plant_Name,Fertility,Photoperiod,Temperature,Rainfall,pH,Light_Hours,Light_Intensity,...,Nitrogen,Phosphorus,Potassium,Yield,Category_pH,Season,N_Ratio,P_Ratio,K_Ratio,Soil_Category
0,clay_118,"rgb(207, 176, 148)",Strawberry,Moderate,Day Neutral,20.887923,747.860765,6.571548,13.091483,533.762876,...,170.800381,118.670058,243.331211,20.369555,low_acidic,Summer,10,10.0,10.0,Loam
1,clay_101,"rgb(213, 143, 92)",Strawberry,Moderate,Day Neutral,18.062721,711.104329,6.251806,13.063016,505.789101,...,179.290364,121.020244,246.910378,20.402751,low_acidic,Spring,10,10.0,10.0,Loam


In [11]:
df['image_color'] = df['image_color'].str.replace('rgb\(|\)', '', regex=True)
df[['R','G','B']] = df['image_color'].str.split(',', expand=True).astype(int)

df['Hue'] = np.arctan2(np.sqrt(3) * (df['G'] - df['B']), 2*df['R'] - df['G'] - df['B'])
df['Saturation'] = 1 - (3 * df[['R','G','B']].min(axis=1)) / (df['R'] + df['G'] + df['B'])
df['Brightness'] = df[['R','G','B']].mean(axis=1)

scaler = MinMaxScaler()
env_features = ['pH','Temperature','Rainfall','Light_Hours']
df[env_features] = scaler.fit_transform(df[env_features])

print("New features:\n", df[['Hue','Saturation','Brightness']].describe())
df[['R','G','B'] + env_features].head(2)

<>:3: SyntaxWarning: invalid escape sequence '\('
<>:3: SyntaxWarning: invalid escape sequence '\('
C:\Users\ERIC TECH RANDA\AppData\Local\Temp\ipykernel_12556\3157847517.py:3: SyntaxWarning: invalid escape sequence '\('
  df['image_color'] = df['image_color'].str.replace('rgb\(|\)', '', regex=True)


New features:
                 Hue    Saturation    Brightness
count  14700.000000  12696.000000  14700.000000
mean       0.124176      0.388990     96.241859
std        0.765013      0.392455     92.522534
min       -3.050683      0.000000      0.000000
25%        0.000000      0.099042      2.666667
50%        0.000000      0.208054     63.333333
75%        0.547321      1.000000    163.000000
max        3.141593      1.000000    255.000000


,R,G,B,pH,Temperature,Rainfall,Light_Hours
0,207,176,148,0.536852,0.377381,0.161065,0.740043
1,213,143,92,0.434840,0.284928,0.143546,0.737425


In [13]:
le_soil = LabelEncoder()
le_plant = LabelEncoder()

df['Soil_Category_Encoded'] = le_soil.fit_transform(df['Soil_Category'])
df['Plant_Name_Encoded'] = le_plant.fit_transform(df['Plant_Name'])

joblib.dump(le_soil, 'soil_encoder.pkl')
joblib.dump(le_plant, 'plant_encoder.pkl')

print("Soil classes:", le_soil.classes_[:5])
print("Plant classes:", le_plant.classes_[:5])
print("\nEncoded sample:")
df[['Soil_Category','Soil_Category_Encoded','Plant_Name','Plant_Name_Encoded']].sample(3)

Soil classes: ['Loam' 'Sandy Loam']
Plant classes: ['Arugula' 'Asparagus' 'Beet' 'Broccoli' 'Cabbage']

Encoded sample:


,Soil_Category,Soil_Category_Encoded,Plant_Name,Plant_Name_Encoded
9223,Sandy Loam,1,Watermelon,20
6217,Loam,0,Chilli Peppers,7
7024,Loam,0,Potatoes,15


In [15]:
features = ['R','G','B','Hue','Saturation','Brightness'] + env_features
X = df[features]
y_soil = df['Soil_Category_Encoded']
y_plant = df['Plant_Name_Encoded']

X_train, X_test, y_soil_train, y_soil_test, y_plant_train, y_plant_test = \
    train_test_split(X, y_soil, y_plant, test_size=0.2, random_state=42)

print("Training shapes:", X_train.shape, y_plant_train.shape)
print("Test shapes:", X_test.shape, y_plant_test.shape)

Training shapes: (11760, 10) (11760,)
Test shapes: (2940, 10) (2940,)


In [17]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

plant_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le_plant.classes_),
    random_state=42
)

grid_search = GridSearchCV(plant_model, param_grid, cv=3, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_plant_train)

plant_model = grid_search.best_estimator_
joblib.dump(plant_model, 'plant_model.pkl')

print("Best parameters:", grid_search.best_params_)
print("Validation accuracy:", grid_search.best_score_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Validation accuracy: 0.9122448979591837


In [75]:
def predict_plant(R, G, B, pH=6.5, temp=25, rainfall=150, light_hours=12, top_n=3):
    hue = np.arctan2(np.sqrt(3) * (G - B), 2*R - G - B)
    sat = 1 - (3 * min(R,G,B)) / (R + G + B + 1e-6)
    bright = (R + G + B) / 3
    
    input_df = pd.DataFrame([[
        R, G, B, hue, sat, bright, pH, temp, rainfall, light_hours
    ]], columns=['R','G','B','Hue','Saturation','Brightness',
                'pH','Temperature','Rainfall','Light_Hours'])
    
    env_features = ['pH','Temperature','Rainfall','Light_Hours']
    input_df[env_features] = scaler.transform(input_df[env_features])
    
    probas = plant_model.predict_proba(input_df)[0]
    top_idx = np.argsort(probas)[-top_n:][::-1]
    
    percentages = (probas[top_idx] * 100).round(1)
    
    plant_names = le_plant.inverse_transform(top_idx)
    
    output = {
        "Analysis": {
            "Soil_RGB": f"{R}, {G}, {B}",
            "pH": pH,
            "Temperature": f"{temp}°C",
            "Rainfall": f"{rainfall}mm",
            "Light_Hours": f"{light_hours}h"
        },
        "Plant_Recommendations": [
            {"Rank": 1, "Plant": plant_names[0], "Confidence": f"{percentages[0]}%"},
            {"Rank": 2, "Plant": plant_names[1], "Confidence": f"{percentages[1]}%"},
            {"Rank": 3, "Plant": plant_names[2], "Confidence": f"{percentages[2]}%"}
        ],
        "Final_Recommendation": {
            "Plant": plant_names[0],
            "Confidence": f"{percentages[0]}%",
            "Message": f"Based on analysis, we recommend planting {plant_names[0]} (Confidence: {percentages[0]}%)"
        }
    }
    
    print("🌱 SOIL ANALYSIS REPORT 🌱")
    print(f"🔍 Soil Color: RGB({R}, {G}, {B})")
    print(f"🧪 pH: {pH} | 🌡️ Temp: {temp}°C | 💧 Rainfall: {rainfall}mm | ☀️ Light: {light_hours}h\n")
    
    print("TOP PLANT OPTIONS:")
    for i in range(top_n):
        print(f"{i+1}. {plant_names[i]} ({percentages[i]}%)")
    
    print("\n⭐ FINAL RECOMMENDATION ⭐")
    print(f"Plant: {plant_names[0]}")
    print(f"Confidence: {percentages[0]}%")
    print(f"\n💡 Recommendation: {plant_names[0]} is the optimal choice for this soil")
    
    return output

print("=== TEST 1 ===")
result = predict_plant(R=120, G=80, B=50)
print("\nReturned Dictionary:")
print(result)

print("\n=== TEST 2 ===")
predict_plant(R=0, G=255, B=0)

=== TEST 1 ===
🌱 SOIL ANALYSIS REPORT 🌱
🔍 Soil Color: RGB(120, 80, 50)
🧪 pH: 6.5 | 🌡️ Temp: 25°C | 💧 Rainfall: 150mm | ☀️ Light: 12h

TOP PLANT OPTIONS:
1. Tomatoes (97.0%)
2. Chard (3.0%)
3. Chilli Peppers (0.0%)

⭐ FINAL RECOMMENDATION ⭐
Plant: Tomatoes
Confidence: 97.0%

💡 Recommendation: Tomatoes is the optimal choice for this soil

Returned Dictionary:
{'Analysis': {'Soil_RGB': '120, 80, 50', 'pH': 6.5, 'Temperature': '25°C', 'Rainfall': '150mm', 'Light_Hours': '12h'}, 'Plant_Recommendations': [{'Rank': 1, 'Plant': 'Tomatoes', 'Confidence': '97.0%'}, {'Rank': 2, 'Plant': 'Chard', 'Confidence': '3.0%'}, {'Rank': 3, 'Plant': 'Chilli Peppers', 'Confidence': '0.0%'}], 'Final_Recommendation': {'Plant': 'Tomatoes', 'Confidence': '97.0%', 'Message': 'Based on analysis, we recommend planting Tomatoes (Confidence: 97.0%)'}}

=== TEST 2 ===
🌱 SOIL ANALYSIS REPORT 🌱
🔍 Soil Color: RGB(0, 255, 0)
🧪 pH: 6.5 | 🌡️ Temp: 25°C | 💧 Rainfall: 150mm | ☀️ Light: 12h

TOP PLANT OPTIONS:
1. Tomatoes (83.

{'Analysis': {'Soil_RGB': '0, 255, 0',
  'pH': 6.5,
  'Temperature': '25°C',
  'Rainfall': '150mm',
  'Light_Hours': '12h'},
 'Plant_Recommendations': [{'Rank': 1,
   'Plant': 'Tomatoes',
   'Confidence': '83.4000015258789%'},
  {'Rank': 2, 'Plant': 'Chard', 'Confidence': '16.100000381469727%'},
  {'Rank': 3,
   'Plant': 'Chilli Peppers',
   'Confidence': '0.20000000298023224%'}],
 'Final_Recommendation': {'Plant': 'Tomatoes',
  'Confidence': '83.4000015258789%',
  'Message': 'Based on analysis, we recommend planting Tomatoes (Confidence: 83.4000015258789%)'}}